# 02_Modeling: ウォークフォワード検証

このノートブックでは、設計仕様に基づきホライゾン別（1d, 3d, 5d）のモデルを訓練し、パージ付きウォークフォワード検証を行います。

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os

# src読み込み用パス設定
sys.path.append('..')
from src.processing import load_and_clean_data
from src.features import generate_features
from src.pooling import pool_boj_data
from src.modeling import walk_forward_validation, calculate_metrics

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 6)

## 1. データ準備

In [ ]:
excel_path = '../data/BOJ_data.xlsx'
meeting_path = '../data/BOJ_meeting_history.csv'

df_clean = load_and_clean_data(excel_path, meeting_path)
df_feat = generate_features(df_clean)
df_pooled = pool_boj_data(df_feat)

print(f"Pooled Data Shape: {df_pooled.shape}")

## 2. ウォークフォワード検証の実行

各ホライゾンについて、2024年以降をテスト期間として検証を行います。

In [ ]:
horizons = [1, 3, 5]
results = {}
start_date = '2024-01-01'

for h in horizons:
    target_col = f'Target_{h}d'
    print(f"\n--- Running Validation for {target_col} ---")
    res = walk_forward_validation(df_pooled, target_col, start_date=start_date)
    results[h] = res

## 3. 全体パフォーマンス評価

In [ ]:
summary_list = []
for h, res in results.items():
    if res.empty:
        continue
    m = calculate_metrics(res['Actual'], res['Pred'])
    m['Horizon'] = f'{h}d'
    summary_list.append(m)

df_summary = pd.DataFrame(summary_list).set_index('Horizon')
display(df_summary)

## 4. レジーム別評価 (ゼロ金利 vs 利上げ期)

2024年3月の利上げ前後で評価を分けます。

In [ ]:
break_date = pd.to_datetime('2024-03-19')

regime_results = []
for h, res in results.items():
    if res.empty:
        continue
    
    # NIRP (Before)
    res_nirp = res[res['Date'] < break_date]
    if not res_nirp.empty:
        m = calculate_metrics(res_nirp['Actual'], res_nirp['Pred'])
        m['Horizon'] = f'{h}d'
        m['Regime'] = 'NIRP'
        regime_results.append(m)
        
    # Hiking (After)
    res_hike = res[res['Date'] >= break_date]
    if not res_hike.empty:
        m = calculate_metrics(res_hike['Actual'], res_hike['Pred'])
        m['Horizon'] = f'{h}d'
        m['Regime'] = 'Hiking'
        regime_results.append(m)

df_regime = pd.DataFrame(regime_results).pivot(index='Horizon', columns='Regime')
display(df_regime)

## 5. 予測値 vs 実績値の可視化 (M1)

In [ ]:
for h in horizons:
    res = results[h]
    if res.empty: continue
    
    # M1(Meeting_Index=1)のみ抽出して時系列比較
    m1_res = res[res['Meeting_Index'] == 1].sort_values('Date')
    
    plt.figure(figsize=(15, 5))
    plt.plot(m1_res['Date'], m1_res['Actual'], label='Actual', alpha=0.5)
    plt.plot(m1_res['Date'], m1_res['Pred'], label='Pred', alpha=0.8)
    plt.title(f"M1 Target_{h}d: Actual vs Prediction")
    plt.legend()
    plt.grid(True)
    plt.show()